# Proteus Arc — EEG-Based Alzheimer's Disease Classification

---

## A Comparative Study of Traditional Machine Learning and Deep Learning for EEG-Based Diagnosis

**Author:** Sai Chintamani  
**Dataset:** OpenNeuro ds004504 (Miltiadous et al., 2023)  
**License:** CC0 (Public Domain)  
**Date:** 2026

---

### Abstract

This notebook implements a complete end-to-end pipeline for classifying electroencephalography (EEG) recordings into three diagnostic categories: **Alzheimer's Disease (AD)**, **Frontotemporal Dementia (FTD)**, and **Cognitively Normal (CN)** controls. We systematically compare two approaches:

- **Candidate 1** — Traditional machine learning classifiers (Logistic Regression, Support Vector Machine, Random Forest) operating on hand-crafted time-domain, frequency-domain, and Hjorth features.
- **Candidate 2** — A compact 1D Convolutional Neural Network (CNN) learning representations directly from raw EEG windows.

All experiments enforce strict **participant-level data splitting** to prevent information leakage and ensure that reported metrics reflect genuine generalization to unseen individuals.

### Research Question

> *"Can a compact 1D CNN learn useful Alzheimer's-related patterns directly from EEG signals and generalize to previously unseen participants better than conventional machine-learning models using engineered EEG features?"*

### Key Design Principles

| Principle | Implementation |
|-----------|---------------|
| No data leakage | Participant-level `GroupShuffleSplit` with disjointness assertions |
| No fabricated results | Notebook halts with `FileNotFoundError` if dataset is missing |
| Reproducibility | Fixed random seed (42) across all stochastic operations |
| Class imbalance handling | Balanced class weights in all models |
| Fair comparison | Both candidates evaluated on identical test participants |

> **⚠️ Disclaimer:** This is a research prototype, not a clinically validated diagnostic system. Performance on this dataset alone does not establish clinical diagnostic reliability.


## Pipeline Architecture Overview

The Proteus Arc pipeline consists of six sequential stages, from raw EEG ingestion through to comparative evaluation. The diagram below illustrates the complete data flow.

### Stage Descriptions

| Stage | Description | Output |
|-------|-------------|--------|
| **1. Data Ingestion** | Load OpenNeuro ds004504 derivatives (`.set` files) via MNE-Python | Raw `mne.io.Raw` objects |
| **2. Preprocessing** | Bandpass filter (0.5–45 Hz, 4th-order Butterworth IIR) | Filtered continuous signals |
| **3. Quality Control** | Check for NaN/Inf values and flat channels | Validated signals |
| **4. Windowing** | Segment into 30-second epochs with 50% overlap | `(n_windows, 19, 15000)` arrays |
| **5. Modeling** | Train Candidate 1 (feature-based ML) and Candidate 2 (1D CNN) | Trained models |
| **6. Evaluation** | Window-level and subject-level metrics with confusion matrices | Performance comparison |


In [ ]:
from IPython.display import Image, display, HTML
try:
    display(Image(filename='notebooks/images/pipeline_architecture.png', width=900))
except FileNotFoundError:
    display(HTML('<p style="color:gray;">Image not available in this environment: notebooks/images/pipeline_architecture.png</p>'))
print('Figure 1: Proteus Arc — End-to-End Pipeline Architecture')

## 1. Environment Inspection

In [1]:
import sys
import platform
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Python: 3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
PyTorch: 2.11.0+cpu
CUDA available: False
Using device: cpu


## 2. Dependency Installation

In [2]:
!pip install -q mne openneuro-py scikit-learn pandas scipy pyyaml joblib matplotlib seaborn


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Repository Access

**Option A:** Clone from GitHub (update the URL to your repository).  
**Option B:** Upload the project `.zip` manually and extract it.  

After this cell, the working directory must contain `src/`, `configs/`, `scripts/`, etc.

In [3]:
import os
import sys
import zipfile

ZIP_PATH = "/content/proteus_arc.zip"
PROJECT_ROOT = "/content/proteus_arc"
required_dirs = ["src", "configs", "scripts", "tests"]

# If the project is not already extracted, extract the uploaded ZIP.
if not os.path.isdir(PROJECT_ROOT) or not all(
    os.path.isdir(os.path.join(PROJECT_ROOT, d)) for d in required_dirs
):
    if not os.path.isfile(ZIP_PATH):
        raise FileNotFoundError(
            "Project ZIP not found at /content/proteus_arc.zip. "
            "Upload proteus_arc.zip using the Colab Files panel, then rerun this cell."
        )
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        # Guard against paths escaping the extraction directory.
        base = os.path.abspath(PROJECT_ROOT)
        for member in archive.infolist():
            destination = os.path.abspath(os.path.join(PROJECT_ROOT, member.filename))
            if os.path.commonpath([base, destination]) != base:
                raise ValueError(f"Unsafe path in ZIP archive: {member.filename}")
        archive.extractall(PROJECT_ROOT)

# Find the directory containing the required project folders.
candidates = []
for root, dirs, files in os.walk(PROJECT_ROOT):
    if all(os.path.isdir(os.path.join(root, d)) for d in required_dirs):
        candidates.append(root)
        dirs[:] = []

if not candidates:
    raise FileNotFoundError(
        "Could not locate src/, configs/, scripts/, and tests/ together. "
        "Check the contents of proteus_arc.zip."
    )

PROJECT_ROOT = candidates[0]
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
for d in required_dirs:
    print(f"  {d}/: OK")
print("Proteus Arc project setup complete.")


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/content/proteus_arc'

## Electroencephalography (EEG) Background

### What is EEG?

Electroencephalography (EEG) is a non-invasive neuroimaging technique that measures electrical activity produced by neuronal populations in the cerebral cortex. Voltage fluctuations are recorded from electrodes placed on the scalp surface.

### The International 10-20 System

The dataset uses the **International 10-20 electrode placement system**, a standardized method for describing and applying the location of scalp electrodes. The "10" and "20" refer to the fact that the distances between adjacent electrodes are either 10% or 20% of the total front-to-back or right-to-left distance of the skull.

### Clinical EEG Frequency Bands

EEG oscillations are conventionally decomposed into frequency bands, each associated with distinct cognitive and pathological states:

| Band | Frequency Range | Physiological Association | AD Relevance |
|------|----------------|--------------------------|--------------|
| **Delta (δ)** | 0.5 – 4.0 Hz | Deep sleep, pathological states | ↑ Increased in moderate-severe AD |
| **Theta (θ)** | 4.0 – 8.0 Hz | Drowsiness, memory encoding | ↑ Increased in early AD |
| **Alpha (α)** | 8.0 – 13.0 Hz | Relaxed wakefulness, posterior dominant rhythm | ↓ Decreased in AD (slowing) |
| **Beta (β)** | 13.0 – 30.0 Hz | Active thinking, motor planning | Variable changes |

### EEG Alterations in Alzheimer's Disease

The hallmark EEG finding in AD is a progressive **"slowing"** of the dominant posterior rhythm, characterized by:

1. **Reduced alpha power** — The normal 8–13 Hz posterior dominant rhythm decreases in amplitude and frequency.
2. **Increased theta/delta power** — Pathological slow-wave activity increases, particularly over temporal and parietal regions.
3. **Reduced complexity** — The EEG signal becomes more regular and less complex, reflecting reduced information processing capacity.
4. **Altered coherence** — Functional connectivity between brain regions (measured via coherence) becomes disrupted.

These changes correlate with disease severity as measured by the Mini-Mental State Examination (MMSE).


In [ ]:
from IPython.display import Image, display, HTML
try:
    display(Image(filename='notebooks/images/eeg_montage.png', width=600))
except FileNotFoundError:
    display(HTML('<p style="color:gray;">Image not available in this environment: notebooks/images/eeg_montage.png</p>'))
print('Figure 2: International 10-20 System — 19-Channel EEG Montage Used in ds004504')

## 4. Dataset Access and Path Configuration

Downloads the **derivatives** subset of OpenNeuro ds004504 using `openneuro-py`.  
The derivatives contain the preprocessed `.set` files (band-pass filtered, ASR, ICA applied by the dataset authors).

**License:** CC0 (public domain).  
**Size:** ~2.5 GB for derivatives.

In [ ]:
import os
import subprocess
import sys

DATASET_PATH = "/content/ds004504"
participants_path = os.path.join(DATASET_PATH, "participants.tsv")

if not os.path.isfile(participants_path):
    print("Downloading OpenNeuro ds004504 metadata and derivatives...")
    print("The download may take a while; do not interrupt this cell.")
    os.makedirs(DATASET_PATH, exist_ok=True)

    cmd = [
        sys.executable, "-m", "openneuro", "download",
        "--dataset", "ds004504",
        "--target-dir", DATASET_PATH,
        "--include", "participants.tsv",
        "--include", "participants.json",
        "--include", "dataset_description.json",
        "--include", "derivatives/*",
    ]
    try:
        result = subprocess.run(cmd, check=True, text=True)
    except Exception as exc:
        raise RuntimeError(
            "OpenNeuro dataset download failed. Review the preceding error. "
            "Confirm the installed openneuro-py CLI syntax and dataset availability."
        ) from exc

if not os.path.isfile(participants_path):
    raise FileNotFoundError(
        f"participants.tsv was not found at {participants_path}. "
        "The dataset download did not complete successfully."
    )

derivatives_path = os.path.join(DATASET_PATH, "derivatives")
if not os.path.isdir(derivatives_path):
    raise FileNotFoundError(
        f"Expected derivatives directory was not found at {derivatives_path}. "
        "The download may not have included the EEG derivative files required by this notebook."
    )

print("Dataset metadata found:", participants_path)
print("Derivatives directory found:", derivatives_path)


## Dataset: OpenNeuro ds004504

### Citation

> Miltiadous, A., Tzimourta, K.D., Afrantou, T., Ioannidis, P., Grigoriadis, N., Tsalikakis, D.G., Angelidis, P., Tsipouras, M.G., Glavas, E., Giannakeas, N. and Tzallas, A.T. (2023). *A dataset of scalp EEG recordings of Alzheimer's disease, frontotemporal dementia and healthy subjects from routine EEG.* Data, 8(6), p.95.

### Cohort Summary

| Property | Value |
|----------|-------|
| Total participants | 88 |
| Alzheimer's Disease (AD) | 36 participants |
| Frontotemporal Dementia (FTD) | 23 participants |
| Cognitively Normal (CN) | 29 participants |
| EEG system | 19-channel, 10-20 montage |
| Sampling rate | 500 Hz |
| Recording condition | Eyes-closed resting state |
| Preprocessing (by authors) | Band-pass filtering, ASR artifact removal, ICA |
| License | CC0 (Public Domain) |

### Pre-existing Preprocessing

The dataset authors applied:
1. **Band-pass filtering** — Initial frequency-domain cleaning.
2. **Artifact Subspace Reconstruction (ASR)** — Removes high-amplitude transient artifacts.
3. **Independent Component Analysis (ICA)** — Removes stereotyped artifacts (eye blinks, muscle activity, line noise).

> **⚠️ Known Limitation:** ASR and ICA were applied to the continuous recordings before segmentation. This means the artifact rejection models were fitted on the full recording, which includes data that may end up in different splits. This is inherent to the dataset and cannot be corrected without access to the raw, unprocessed recordings. This limitation must be acknowledged in any research report.


## 5. Dataset Audit

In [ ]:
import pandas as pd
import yaml

participants_df = pd.read_csv(os.path.join(DATASET_PATH, 'participants.tsv'), sep='\t')
print(f'Total participants: {len(participants_df)}')
print(f'Columns: {list(participants_df.columns)}')
print(f'\nClass distribution (Group column):')
print(participants_df['Group'].value_counts())
print(f'\nGender distribution:')
print(participants_df['Gender'].value_counts())
print(f'\nAge statistics:')
print(participants_df['Age'].describe())
print(f'\nMMSE statistics by group:')
print(participants_df.groupby('Group')['MMSE'].describe())

# Validate against project config
with open('configs/data.yaml', 'r') as f:
    data_config = yaml.safe_load(f)['dataset']

expected_classes = set(data_config['classes'])
actual_classes = set(participants_df['Group'].unique())
print(f'\nExpected classes: {expected_classes}')
print(f'Actual classes:   {actual_classes}')
if expected_classes != actual_classes:
    print(f'WARNING: Class mismatch! Expected {expected_classes}, found {actual_classes}')
else:
    print('Class labels match configuration.')

# Check for missing or duplicate participant IDs
dupes = participants_df['participant_id'].duplicated().sum()
nulls = participants_df['participant_id'].isnull().sum()
print(f'\nDuplicate participant_ids: {dupes}')
print(f'Null participant_ids: {nulls}')
if dupes > 0 or nulls > 0:
    raise ValueError('Participant metadata has duplicates or nulls. Cannot proceed.')

## 6. Participant and Class-Distribution Inspection

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class distribution
label_map = data_config['mapping']
participants_df['Diagnosis'] = participants_df['Group'].map(label_map)
sns.countplot(data=participants_df, x='Diagnosis', ax=axes[0], palette='Set2')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')

# Age distribution by group
sns.boxplot(data=participants_df, x='Diagnosis', y='Age', ax=axes[1], palette='Set2')
axes[1].set_title('Age by Diagnosis')

# MMSE by group
sns.boxplot(data=participants_df, x='Diagnosis', y='MMSE', ax=axes[2], palette='Set2')
axes[2].set_title('MMSE by Diagnosis')

plt.tight_layout()
os.makedirs('outputs/figures', exist_ok=True)
plt.savefig('outputs/figures/cohort_demographics.png', dpi=150)
plt.show()
print('Saved: outputs/figures/cohort_demographics.png')

## 7. EEG Preprocessing — Load, Verify, Filter, Window

Loads each participant's preprocessed `.set` file from `derivatives/`.  
Verifies actual channel count and sampling rate per recording.  
Applies the project's bandpass filter (0.5–45 Hz) and windows into 30 s epochs with 50% overlap.

In [ ]:
import mne
import numpy as np
from src.proteus_arc.preprocessing.filtering import EEGFilter
from src.proteus_arc.preprocessing.windowing import WindowGenerator
from src.proteus_arc.preprocessing.quality_control import QualityControl

mne.set_log_level('ERROR')

with open('configs/preprocessing.yaml', 'r') as f:
    preproc_config = yaml.safe_load(f)

eeg_filter = EEGFilter(
    l_freq=preproc_config['filtering']['l_freq'],
    h_freq=preproc_config['filtering']['h_freq']
)
window_gen = WindowGenerator(
    epoch_duration_sec=preproc_config['windowing']['epoch_duration_sec'],
    overlap_ratio=preproc_config['windowing']['overlap_ratio']
)

expected_channels = data_config['channels']
expected_sfreq = data_config['sampling_rate']

all_windows = []
all_labels = []
all_groups = []
skipped = []
recording_log = []

derivatives_dir = os.path.join(DATASET_PATH, 'derivatives')
if not os.path.isdir(derivatives_dir):
    raise FileNotFoundError(
        f'Derivatives directory not found at {derivatives_dir}. '
        'Ensure the dataset download included derivatives/*.'
    )

for _, row in participants_df.iterrows():
    pid = row['participant_id']
    group = row['Group']
    label = label_map.get(group, group)

    eeg_file = os.path.join(derivatives_dir, pid, 'eeg', f'{pid}_task-eyesclosed_eeg.set')
    if not os.path.exists(eeg_file):
        skipped.append({'participant_id': pid, 'reason': 'File not found'})
        continue

    try:
        raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose='ERROR')
    except Exception as e:
        skipped.append({'participant_id': pid, 'reason': str(e)})
        continue

    actual_sfreq = raw.info['sfreq']
    actual_channels = raw.ch_names
    duration_sec = raw.times[-1] if len(raw.times) > 0 else 0.0

    # Channel selection: pick only the expected 19 channels if present
    available = [ch for ch in expected_channels if ch in actual_channels]
    if len(available) < len(expected_channels):
        missing_ch = set(expected_channels) - set(available)
        skipped.append({'participant_id': pid, 'reason': f'Missing channels: {missing_ch}'})
        continue
    raw.pick_channels(expected_channels, ordered=True)

    # Resample if needed
    if actual_sfreq != expected_sfreq:
        raw.resample(expected_sfreq)
        actual_sfreq = expected_sfreq

    data = raw.get_data()  # (channels, samples)

    # Quality checks
    if not QualityControl.check_nan_inf(data):
        skipped.append({'participant_id': pid, 'reason': 'NaN/Inf in signal'})
        continue
    flat = QualityControl.identify_flat_channels(data)
    if flat:
        skipped.append({'participant_id': pid, 'reason': f'Flat channels: {flat}'})
        continue

    # Apply bandpass filter
    try:
        data = eeg_filter.apply(data, actual_sfreq)
    except ValueError as e:
        skipped.append({'participant_id': pid, 'reason': f'Filter error: {e}'})
        continue

    # Windowing
    metadata = {'participant_id': pid, 'label': label, 'sfreq': actual_sfreq}
    windows, win_meta = window_gen.generate_windows(data, actual_sfreq, metadata)

    if windows.shape[0] == 0:
        skipped.append({'participant_id': pid, 'reason': 'Recording too short for windowing'})
        continue

    recording_log.append({
        'participant_id': pid, 'label': label, 'sfreq': actual_sfreq,
        'n_channels': data.shape[0], 'duration_sec': duration_sec,
        'n_windows': windows.shape[0]
    })

    all_windows.append(windows)
    all_labels.extend([label] * windows.shape[0])
    all_groups.extend([pid] * windows.shape[0])

    del raw, data, windows  # free memory

if len(all_windows) == 0:
    raise RuntimeError('No valid recordings were loaded. Check the dataset and skipped list.')

X_all = np.concatenate(all_windows, axis=0)
y_all = np.array(all_labels)
groups_all = np.array(all_groups)

print(f'\n=== Data Loading Summary ===')
print(f'Total windows: {X_all.shape[0]}')
print(f'Window shape:  {X_all.shape[1:]}  (channels, samples)')
print(f'Unique participants loaded: {len(np.unique(groups_all))}')
print(f'Skipped recordings: {len(skipped)}')
print(f'Label distribution: {dict(zip(*np.unique(y_all, return_counts=True)))}')

if skipped:
    print('\nSkipped details:')
    for s in skipped:
        print(f"  {s['participant_id']}: {s['reason']}")

# Save recording log
rec_log_df = pd.DataFrame(recording_log)
os.makedirs('outputs/audit', exist_ok=True)
rec_log_df.to_csv('outputs/audit/recording_log.csv', index=False)
pd.DataFrame(skipped).to_csv('outputs/audit/skipped_recordings.csv', index=False)
print('\nSaved: outputs/audit/recording_log.csv, outputs/audit/skipped_recordings.csv')

## 8. Signal-Quality Checks

In [ ]:
# Aggregate quality stats across all loaded windows
print('Signal quality across all loaded windows:')
print(f'  All finite: {np.isfinite(X_all).all()}')
print(f'  Global min: {X_all.min():.6f}')
print(f'  Global max: {X_all.max():.6f}')
print(f'  Global std: {X_all.std():.6f}')

# Per-channel variance check
channel_vars = X_all.reshape(-1, X_all.shape[1], X_all.shape[2]).var(axis=(0, 2))
print(f'\nPer-channel variance (across all windows):')
for i, ch in enumerate(expected_channels):
    print(f'  {ch}: {channel_vars[i]:.6f}')

## 9. Window Generation Summary

Windows were generated in Cell 7. This cell provides a summary view.

In [ ]:
print(f'Window configuration:')
print(f'  Duration: {preproc_config["windowing"]["epoch_duration_sec"]} sec')
print(f'  Overlap:  {preproc_config["windowing"]["overlap_ratio"] * 100}%')
print(f'  Samples per window: {X_all.shape[2]}')
print(f'  Channels per window: {X_all.shape[1]}')

# Windows per participant
win_per_sub = pd.Series(groups_all).value_counts().sort_index()
print(f'\nWindows per participant:')
print(f'  Min: {win_per_sub.min()}, Max: {win_per_sub.max()}, Mean: {win_per_sub.mean():.1f}')

fig, ax = plt.subplots(figsize=(14, 3))
win_per_sub.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Windows per Participant')
ax.set_ylabel('Number of Windows')
plt.tight_layout()
plt.savefig('outputs/figures/windows_per_participant.png', dpi=150)
plt.show()

## Participant-Level Splitting and Data Leakage Prevention

### The Data Leakage Problem in EEG

In EEG studies, each participant typically produces many windows (epochs). If we perform a naive random train/test split at the **window level**, windows from the same participant will appear in both the training and test sets. The model may then learn **participant-specific artifacts** (electrode impedance, scalp thickness, habitual muscle tension) rather than **disease-related patterns**.

This is known as **data leakage** and leads to artificially inflated performance metrics that do not reflect real-world generalization.

### Our Solution: GroupShuffleSplit

We use scikit-learn's `GroupShuffleSplit` with participant IDs as the grouping variable. This guarantees that **all windows from a single participant** are assigned to exactly one split.

**Two-Stage Splitting Protocol:**

| Stage | Operation | Result |
|-------|-----------|--------|
| Stage 1 | 80/20 split (train+val vs. test) | Test set: ~18 participants |
| Stage 2 | 75/25 split of remainder (train vs. val) | Train: ~53, Val: ~17 participants |

**Effective proportions:** ~60% Train / ~20% Validation / ~20% Test

### Runtime Assertions

The notebook enforces three explicit assertions after every split:

```python
assert train_participants.isdisjoint(val_participants)   # No train/val overlap
assert train_participants.isdisjoint(test_participants)   # No train/test overlap
assert val_participants.isdisjoint(test_participants)      # No val/test overlap
```

If any assertion fails, the notebook halts immediately with a clear error message identifying the leaked participants.

### Additional Safeguards

| Safeguard | Implementation |
|-----------|---------------|
| Baseline scaling | `StandardScaler` inside `sklearn.Pipeline` — fitted on train only |
| CNN class weights | Computed from `y_train` labels only |
| CNN checkpoint | Selected by **validation loss** only (test data never influences model selection) |
| Baselines use train+val | Appropriate since baselines don't use early stopping |


In [ ]:
from IPython.display import Image, display, HTML
try:
    display(Image(filename='notebooks/images/participant_split.png', width=800))
except FileNotFoundError:
    display(HTML('<p style="color:gray;">Image not available in this environment: notebooks/images/participant_split.png</p>'))
print('Figure 5: Participant-Level Data Splitting — Preventing Information Leakage')

## 10. Participant-Level Train / Validation / Test Splitting

Uses a two-stage `GroupShuffleSplit` to create **mutually exclusive** participant sets:  
- Train: ~60%  
- Validation: ~20%  
- Test: ~20%  

The same split is used for both Candidate 1 (baselines) and Candidate 2 (CNN).

In [ ]:
from sklearn.preprocessing import LabelEncoder
from src.proteus_arc.evaluation.participant_split import ParticipantSplitter

le = LabelEncoder()
y_encoded = le.fit_transform(y_all)
class_names = list(le.classes_)
print(f'Label encoding: {dict(zip(class_names, le.transform(class_names)))}')

RANDOM_SEED = 42

# Stage 1: Split off Test
splitter_test = ParticipantSplitter(method='group_shuffle', test_size=0.2, random_state=RANDOM_SEED)
X_temp, X_test, y_temp, y_test, g_temp, g_test = splitter_test.split(X_all, y_encoded, groups_all)

# Stage 2: Split Train / Val from remainder
splitter_val = ParticipantSplitter(method='group_shuffle', test_size=0.25, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val, g_train, g_val = splitter_val.split(X_temp, y_temp, g_temp)

# Leakage assertions
train_p = set(g_train)
val_p = set(g_val)
test_p = set(g_test)

assert train_p.isdisjoint(val_p), f'LEAKAGE: train/val overlap: {train_p & val_p}'
assert train_p.isdisjoint(test_p), f'LEAKAGE: train/test overlap: {train_p & test_p}'
assert val_p.isdisjoint(test_p), f'LEAKAGE: val/test overlap: {val_p & test_p}'

print(f'\n=== Participant-Level Split ===')
print(f'Train:      {len(train_p):>3} participants, {len(y_train):>5} windows')
print(f'Validation: {len(val_p):>3} participants, {len(y_val):>5} windows')
print(f'Test:       {len(test_p):>3} participants, {len(y_test):>5} windows')
print(f'Total:      {len(train_p)+len(val_p)+len(test_p)} participants')

# Class balance per split
for name, yy in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    counts = dict(zip(*np.unique(yy, return_counts=True)))
    decoded = {le.inverse_transform([k])[0]: v for k, v in counts.items()}
    print(f'  {name} labels: {decoded}')

# Save split manifest
split_manifest = {
    'train_participants': sorted(list(train_p)),
    'val_participants': sorted(list(val_p)),
    'test_participants': sorted(list(test_p)),
    'random_seed': RANDOM_SEED
}
with open('outputs/audit/split_manifest.yaml', 'w') as f:
    yaml.dump(split_manifest, f)
print('\nSaved: outputs/audit/split_manifest.yaml')

## Feature Engineering Theory (Candidate 1)

Candidate 1 extracts hand-crafted features from each EEG window across three domains. For each of the 19 channels, we compute features independently, resulting in a feature vector of dimensionality $19 \times N_{features}$.

### 1. Time-Domain Features

These features capture the statistical properties of the raw EEG amplitude distribution:

| Feature | Formula | Interpretation |
|---------|---------|----------------|
| **Mean** | $\bar{x} = \frac{1}{N}\sum_{i=1}^{N} x_i$ | DC offset (typically ≈ 0 after filtering) |
| **Standard Deviation** | $\sigma = \sqrt{\frac{1}{N-1}\sum_{i=1}^{N}(x_i - \bar{x})^2}$ | Signal amplitude variability |
| **Variance** | $\sigma^2$ | Mean power of the signal |
| **Root Mean Square** | $\text{RMS} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} x_i^2}$ | Effective signal amplitude |
| **Skewness** | $\gamma_1 = \frac{1}{N}\sum_{i=1}^{N}\left(\frac{x_i - \bar{x}}{\sigma}\right)^3$ | Asymmetry of amplitude distribution |
| **Kurtosis** | $\kappa = \frac{1}{N}\sum_{i=1}^{N}\left(\frac{x_i - \bar{x}}{\sigma}\right)^4 - 3$ | Tail heaviness (transient detection) |

### 2. Frequency-Domain Features (Welch's Method)

We estimate the **Power Spectral Density (PSD)** using Welch's method, which divides the signal into overlapping segments, computes the periodogram of each, and averages them to reduce variance.

For each clinical EEG band $b$ with frequency range $[f_{low}, f_{high}]$:

$$P_b = \int_{f_{low}}^{f_{high}} S_{xx}(f) \, df$$

where $S_{xx}(f)$ is the PSD estimate. We compute **absolute band power** for:

| Band | Range | Clinical Significance in AD |
|------|-------|----------------------------|
| Delta ($\delta$) | 0.5 – 4.0 Hz | Pathological slowing marker |
| Theta ($\theta$) | 4.0 – 8.0 Hz | Early cognitive decline indicator |
| Alpha ($\alpha$) | 8.0 – 13.0 Hz | Posterior dominant rhythm integrity |
| Beta ($\beta$) | 13.0 – 30.0 Hz | Cortical activation level |

### 3. Hjorth Parameters

Hjorth parameters (Hjorth, 1970) provide a compact, computationally efficient description of the signal's temporal structure using only variance computations:

| Parameter | Formula | Interpretation |
|-----------|---------|----------------|
| **Activity** | $H_0 = \text{var}(x)$ | Total signal power |
| **Mobility** | $H_1 = \sqrt{\frac{\text{var}(x')}{\text{var}(x)}}$ | Mean frequency (proportion of standard deviation of the power spectrum) |
| **Complexity** | $H_2 = \frac{H_1(x')}{H_1(x)}$ | Bandwidth — how similar the signal is to a pure sine wave |

where $x'$ denotes the first temporal derivative of the signal.


In [ ]:
from IPython.display import Image, display, HTML
try:
    display(Image(filename='notebooks/images/feature_extraction.png', width=800))
except FileNotFoundError:
    display(HTML('<p style="color:gray;">Image not available in this environment: notebooks/images/feature_extraction.png</p>'))
print('Figure 3: Feature Extraction Pipeline — Time Domain, Frequency Domain, and Hjorth Parameters')

## 11. Baseline Feature Extraction (Candidate 1)

In [ ]:
from src.proteus_arc.features.feature_pipeline import FeaturePipeline

feat_pipeline = FeaturePipeline(
    config_path='configs/baseline.yaml',
    sfreq=float(expected_sfreq),
    channel_names=expected_channels
)

print('Extracting features for training set...')
X_train_feat = feat_pipeline.transform(X_train).to_numpy(dtype=float)
print(f'  Train features shape: {X_train_feat.shape}')

print('Extracting features for validation set...')
X_val_feat = feat_pipeline.transform(X_val).to_numpy(dtype=float)

print('Extracting features for test set...')
X_test_feat = feat_pipeline.transform(X_test).to_numpy(dtype=float)

feature_names = list(feat_pipeline.transform(X_train[:1]).columns)
print(f'  Total features per window: {len(feature_names)}')
print(f'  Feature names (first 10): {feature_names[:10]}')

## Traditional Machine Learning Classifiers (Candidate 1)

All three classifiers are wrapped in `sklearn.Pipeline` objects containing:
1. **SimpleImputer** (median strategy) — Handles any NaN values from feature extraction.
2. **StandardScaler** — Zero-mean, unit-variance normalization fitted **only on training data**.
3. **Classifier** — The actual classification algorithm.

### 1. Logistic Regression (LR)

A linear model that estimates the posterior probability of each class using the softmax function:

$$P(y=k | \mathbf{x}) = \frac{\exp(\mathbf{w}_k^\top \mathbf{x} + b_k)}{\sum_{j=1}^{K} \exp(\mathbf{w}_j^\top \mathbf{x} + b_j)}$$

- **Purpose:** Establishes a linear separability baseline.
- **Regularization:** L2 penalty (default).
- **Class balancing:** `class_weight='balanced'` — inversely proportional to class frequency.

### 2. Support Vector Machine (SVM)

An SVM with a Radial Basis Function (RBF) kernel maps the features into an infinite-dimensional space to find a maximum-margin hyperplane:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2\right)$$

- **Purpose:** Tests whether non-linear feature interactions improve classification.
- **Kernel:** RBF with `gamma='scale'`.
- **Probability calibration:** Enabled via Platt scaling for ROC-AUC computation.

### 3. Random Forest (RF)

An ensemble of 300 decision trees, each trained on a bootstrap sample with random feature subsets:

$$\hat{y} = \text{mode}\left\{h_t(\mathbf{x})\right\}_{t=1}^{T}$$

where $h_t$ is the $t$-th decision tree.

- **Purpose:** Captures complex non-linear interactions and provides feature importance rankings.
- **Configuration:** `n_estimators=300`, `min_samples_leaf=2`, `class_weight='balanced'`.


## 12. Train Logistic Regression

In [ ]:
import time
import joblib
from src.proteus_arc.models.baseline_models import BaselineModels
from src.proteus_arc.evaluation.metrics import Evaluator

os.makedirs('outputs/checkpoints', exist_ok=True)
os.makedirs('outputs/metrics', exist_ok=True)

baseline_results = {}

# Combine train+val for baselines (they don't use early stopping on val)
X_trainval_feat = np.concatenate([X_train_feat, X_val_feat], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)
g_trainval = np.concatenate([g_train, g_val], axis=0)

bm = BaselineModels(config_path='configs/baseline.yaml')

# --- Logistic Regression ---
lr_model = bm.get_logistic_regression()
t0 = time.time()
lr_model.fit(X_trainval_feat, y_trainval)
lr_train_time = time.time() - t0

t0 = time.time()
lr_pred = lr_model.predict(X_test_feat)
lr_proba = lr_model.predict_proba(X_test_feat)
lr_infer_time = time.time() - t0

lr_metrics = Evaluator.evaluate_model(y_test, lr_pred, lr_proba, classes=list(range(len(class_names))))
lr_metrics['Train Time (s)'] = lr_train_time
lr_metrics['Inference Time (s)'] = lr_infer_time
baseline_results['Logistic Regression'] = lr_metrics

joblib.dump(lr_model, 'outputs/checkpoints/logistic_regression.joblib')
print(f"Logistic Regression — Acc: {lr_metrics['Accuracy']:.4f}, F1: {lr_metrics['Macro F1']:.4f}, Train: {lr_train_time:.2f}s")

## 13. Train SVM

In [ ]:
svm_model = bm.get_svm()
t0 = time.time()
svm_model.fit(X_trainval_feat, y_trainval)
svm_train_time = time.time() - t0

t0 = time.time()
svm_pred = svm_model.predict(X_test_feat)
svm_proba = svm_model.predict_proba(X_test_feat)
svm_infer_time = time.time() - t0

svm_metrics = Evaluator.evaluate_model(y_test, svm_pred, svm_proba, classes=list(range(len(class_names))))
svm_metrics['Train Time (s)'] = svm_train_time
svm_metrics['Inference Time (s)'] = svm_infer_time
baseline_results['SVM'] = svm_metrics

joblib.dump(svm_model, 'outputs/checkpoints/svm.joblib')
print(f"SVM — Acc: {svm_metrics['Accuracy']:.4f}, F1: {svm_metrics['Macro F1']:.4f}, Train: {svm_train_time:.2f}s")

## 14. Train Random Forest

In [ ]:
rf_model = bm.get_random_forest()
t0 = time.time()
rf_model.fit(X_trainval_feat, y_trainval)
rf_train_time = time.time() - t0

t0 = time.time()
rf_pred = rf_model.predict(X_test_feat)
rf_proba = rf_model.predict_proba(X_test_feat)
rf_infer_time = time.time() - t0

rf_metrics = Evaluator.evaluate_model(y_test, rf_pred, rf_proba, classes=list(range(len(class_names))))
rf_metrics['Train Time (s)'] = rf_train_time
rf_metrics['Inference Time (s)'] = rf_infer_time
baseline_results['Random Forest'] = rf_metrics

joblib.dump(rf_model, 'outputs/checkpoints/random_forest.joblib')
print(f"Random Forest — Acc: {rf_metrics['Accuracy']:.4f}, F1: {rf_metrics['Macro F1']:.4f}, Train: {rf_train_time:.2f}s")

## 1D Convolutional Neural Network Architecture (Candidate 2)

### Motivation

Unlike Candidate 1, which relies on hand-crafted features requiring domain expertise, Candidate 2 learns hierarchical spatial-temporal representations **directly from the raw EEG signal**. The 1D CNN applies temporal convolutions across the time axis, treating each EEG channel as a separate input "feature map."

### Architecture Summary

```
Input: (batch_size, 19 channels, 15000 samples)
                    │
    ┌───────────────┴───────────────┐
    │     CONVOLUTIONAL BLOCK 1     │
    │  Conv1D(19→16, kernel=31)     │
    │  BatchNorm1D(16)              │
    │  ReLU                         │
    │  MaxPool1D(4) → 3750 samples  │
    └───────────────┬───────────────┘
                    │
    ┌───────────────┴───────────────┐
    │     CONVOLUTIONAL BLOCK 2     │
    │  Conv1D(16→32, kernel=15)     │
    │  BatchNorm1D(32)              │
    │  ReLU                         │
    │  MaxPool1D(4) → 937 samples   │
    └───────────────┬───────────────┘
                    │
    ┌───────────────┴───────────────┐
    │     GLOBAL POOLING            │
    │  AdaptiveAvgPool1D(1) → 1     │
    └───────────────┬───────────────┘
                    │
    ┌───────────────┴───────────────┐
    │     CLASSIFICATION HEAD       │
    │  Dropout(p=0.5)               │
    │  Linear(32 → 3)              │
    │  Output: [AD, CN, FTD]        │
    └───────────────────────────────┘
```

### Design Rationale

| Design Choice | Rationale |
|--------------|-----------|
| **Large initial kernel (31)** | Captures broad temporal patterns (~62 ms at 500 Hz), corresponding to Delta/Theta oscillation morphology |
| **Two convolutional blocks** | Keeps the model compact to prevent overfitting on 88 participants |
| **Batch Normalization** | Stabilizes training and allows higher learning rates |
| **Adaptive Average Pooling** | Produces a fixed-size feature vector regardless of input length |
| **Dropout (p=0.5)** | Strong regularization critical for small biomedical datasets |
| **Weighted Cross-Entropy Loss** | Addresses the class imbalance (AD:36, CN:29, FTD:23) |

### Training Configuration

| Hyperparameter | Value |
|---------------|-------|
| Optimizer | Adam |
| Learning rate | 0.001 |
| Weight decay (L2) | 0.0001 |
| Batch size | 32 |
| Max epochs | 50 |
| Early stopping patience | 10 epochs (on validation loss) |
| Checkpoint selection | Best validation loss |
| Random seed | 42 |


In [ ]:
from IPython.display import Image, display, HTML
try:
    display(Image(filename='notebooks/images/cnn_architecture.png', width=900))
except FileNotFoundError:
    display(HTML('<p style="color:gray;">Image not available in this environment: notebooks/images/cnn_architecture.png</p>'))
print('Figure 4: EEG1DCNN — Compact 1D Convolutional Neural Network Architecture')

## 15. CNN Initialization (Candidate 2)

In [ ]:
from src.proteus_arc.models.cnn_1d import EEG1DCNN

with open('configs/cnn.yaml', 'r') as f:
    cnn_config = yaml.safe_load(f)

cnn_arch = cnn_config['cnn']['architecture']
cnn_train_cfg = cnn_config['cnn']['training']

# Verify architecture matches data dimensions
assert cnn_arch['in_channels'] == X_train.shape[1], \
    f"CNN expects {cnn_arch['in_channels']} channels but data has {X_train.shape[1]}"
assert cnn_arch['num_classes'] == len(class_names), \
    f"CNN expects {cnn_arch['num_classes']} classes but found {len(class_names)}"

model = EEG1DCNN(**cnn_arch)
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Smoke test
dummy = torch.randn(2, cnn_arch['in_channels'], X_train.shape[2])
out = model(dummy)
print(f'Smoke test — Input: {dummy.shape} -> Output: {out.shape}')
assert out.shape == (2, cnn_arch['num_classes'])

## 16. CNN Training and Validation

In [ ]:
from torch.utils.data import DataLoader
from src.proteus_arc.models.training import EEGDataset, CNNTrainer

torch.manual_seed(cnn_train_cfg['random_seed'])
np.random.seed(cnn_train_cfg['random_seed'])

train_dataset = EEGDataset(X_train, y_train, g_train)
val_dataset = EEGDataset(X_val, y_val, g_val)
test_dataset = EEGDataset(X_test, y_test, g_test)

train_loader = DataLoader(train_dataset, batch_size=cnn_train_cfg['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cnn_train_cfg['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=cnn_train_cfg['batch_size'], shuffle=False)

# Compute class weights from TRAINING labels only
class_counts = np.bincount(y_train)
total = len(y_train)
class_weights = torch.tensor([total / max(c, 1) for c in class_counts], dtype=torch.float32)
print(f'Class weights (from train only): {class_weights}')

trainer = CNNTrainer(
    model=model,
    device=DEVICE,
    learning_rate=cnn_train_cfg['learning_rate'],
    weight_decay=cnn_train_cfg['weight_decay'],
    patience=cnn_train_cfg['early_stopping_patience'],
    class_weights=class_weights
)

cnn_save_path = 'outputs/checkpoints/best_cnn.pth'
t0 = time.time()
history = trainer.fit(train_loader, val_loader, epochs=cnn_train_cfg['epochs'], save_path=cnn_save_path)
cnn_train_time = time.time() - t0
print(f'\nTotal CNN training time: {cnn_train_time:.1f}s')

# Save training history
pd.DataFrame(history).to_csv('outputs/metrics/cnn_training_history.csv', index=False)
print('Saved: outputs/metrics/cnn_training_history.csv')

## Evaluation Methodology

### Window-Level Metrics

Each model produces predictions for every 30-second EEG window in the test set. We compute:

| Metric | Formula | Purpose |
|--------|---------|---------|
| **Accuracy** | $\frac{TP + TN}{TP + TN + FP + FN}$ | Overall correctness |
| **Balanced Accuracy** | $\frac{1}{K}\sum_{k=1}^{K} \text{Recall}_k$ | Corrects for class imbalance |
| **Macro Precision** | $\frac{1}{K}\sum_{k=1}^{K} \frac{TP_k}{TP_k + FP_k}$ | Average per-class positive predictive value |
| **Macro Recall (Sensitivity)** | $\frac{1}{K}\sum_{k=1}^{K} \frac{TP_k}{TP_k + FN_k}$ | Average per-class detection rate |
| **Macro F1** | $\frac{1}{K}\sum_{k=1}^{K} \frac{2 \cdot P_k \cdot R_k}{P_k + R_k}$ | Harmonic mean of precision and recall |
| **ROC-AUC** | Area under the one-vs-rest ROC curve | Discrimination ability across thresholds |

### Subject-Level Aggregation (Majority Voting)

For clinical relevance, we aggregate window-level predictions to produce a **single diagnosis per participant**:

$$\hat{y}_{\text{subject}} = \text{mode}\left\{\hat{y}_w : w \in \text{windows}(\text{subject})\right\}$$

This majority-voting approach is more clinically meaningful than window-level metrics because a clinician would make one diagnosis per patient, not per epoch.

### Confusion Matrices

We generate confusion matrices for all four models to visualize:
- Which diagnostic categories are most frequently confused.
- Whether AD vs. FTD discrimination is harder than AD vs. CN.
- Class-specific sensitivity and specificity patterns.


## 17. Final Held-Out Test Evaluation

Loads the **best checkpoint** (selected by validation loss) and evaluates on the held-out test participants.  
Test data was never used for training, validation, early stopping, or model selection.

In [ ]:
# Load best CNN checkpoint
model.load_state_dict(torch.load(cnn_save_path, weights_only=True, map_location=DEVICE))
model.to(DEVICE)

t0 = time.time()
cnn_test_loss, cnn_test_acc, cnn_test_f1, cnn_labels, cnn_preds, cnn_groups = trainer.val_epoch(test_loader)
cnn_infer_time = time.time() - t0

# Get probabilities for ROC-AUC
model.eval()
all_probas = []
with torch.no_grad():
    for X_batch, _, _ in test_loader:
        X_batch = X_batch.to(DEVICE)
        logits = model(X_batch)
        probas = torch.softmax(logits, dim=1)
        all_probas.append(probas.cpu().numpy())
cnn_proba = np.concatenate(all_probas, axis=0)

cnn_metrics = Evaluator.evaluate_model(cnn_labels, cnn_preds, cnn_proba, classes=list(range(len(class_names))))
cnn_metrics['Train Time (s)'] = cnn_train_time
cnn_metrics['Inference Time (s)'] = cnn_infer_time

print(f"CNN (Window-Level) — Acc: {cnn_metrics['Accuracy']:.4f}, F1: {cnn_metrics['Macro F1']:.4f}")

## 18. Participant-Level Aggregation

For each test participant, window-level predictions are aggregated via **majority voting**.

In [ ]:
all_subject_results = {}

# Baseline subject-level
for model_name, preds in [('Logistic Regression', lr_pred), ('SVM', svm_pred), ('Random Forest', rf_pred)]:
    y_true_sub, y_pred_sub = Evaluator.subject_level_vote(y_test, preds, g_test)
    sub_metrics = Evaluator.evaluate_model(y_true_sub, y_pred_sub, classes=list(range(len(class_names))))
    all_subject_results[model_name] = sub_metrics
    print(f'{model_name} (Subject-Level) — Acc: {sub_metrics["Accuracy"]:.4f}, F1: {sub_metrics["Macro F1"]:.4f}')

# CNN subject-level
y_true_sub_cnn, y_pred_sub_cnn = Evaluator.subject_level_vote(cnn_labels, cnn_preds, cnn_groups)
cnn_sub_metrics = Evaluator.evaluate_model(y_true_sub_cnn, y_pred_sub_cnn, classes=list(range(len(class_names))))
all_subject_results['1D CNN'] = cnn_sub_metrics
print(f'1D CNN (Subject-Level) — Acc: {cnn_sub_metrics["Accuracy"]:.4f}, F1: {cnn_sub_metrics["Macro F1"]:.4f}')

## 19. Metrics and Confusion Matrices

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
all_preds_dict = {
    'Logistic Regression': lr_pred,
    'SVM': svm_pred,
    'Random Forest': rf_pred,
    '1D CNN': cnn_preds
}
y_test_for_cm = {'Logistic Regression': y_test, 'SVM': y_test, 'Random Forest': y_test, '1D CNN': cnn_labels}

for ax, (name, preds) in zip(axes.flat, all_preds_dict.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test_for_cm[name], preds,
        display_labels=class_names,
        ax=ax, cmap='Blues', colorbar=False
    )
    ax.set_title(f'{name} (Window-Level)')

plt.tight_layout()
plt.savefig('outputs/figures/confusion_matrices.png', dpi=150)
plt.show()
print('Saved: outputs/figures/confusion_matrices.png')

## 20. Baseline vs. CNN Comparison

In [ ]:
# Window-level comparison table
all_window_metrics = {**baseline_results, '1D CNN': cnn_metrics}

comparison_cols = ['Accuracy', 'Balanced Accuracy', 'Macro F1', 'Macro Precision', 'Macro Recall', 'ROC-AUC', 'Train Time (s)', 'Inference Time (s)']
comparison_data = []
for name, m in all_window_metrics.items():
    row = {'Model': name}
    for col in comparison_cols:
        val = m.get(col, np.nan)
        row[col] = round(val, 4) if isinstance(val, float) else val
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
print('=== Window-Level Comparison ===')
print(comparison_df.to_string(index=False))

# Subject-level comparison table
sub_comparison_data = []
for name, m in all_subject_results.items():
    row = {'Model': name}
    for col in ['Accuracy', 'Balanced Accuracy', 'Macro F1']:
        val = m.get(col, np.nan)
        row[col] = round(val, 4) if isinstance(val, float) else val
    sub_comparison_data.append(row)

sub_comparison_df = pd.DataFrame(sub_comparison_data)
print('\n=== Subject-Level Comparison ===')
print(sub_comparison_df.to_string(index=False))

# Save
comparison_df.to_csv('outputs/metrics/window_level_comparison.csv', index=False)
sub_comparison_df.to_csv('outputs/metrics/subject_level_comparison.csv', index=False)
print('\nSaved comparison tables to outputs/metrics/')

## 21. Error Analysis

In [ ]:
# Save all predictions with participant IDs for error analysis
predictions_df = pd.DataFrame({
    'participant_id': g_test,
    'y_true': y_test,
    'y_true_label': le.inverse_transform(y_test),
    'lr_pred': lr_pred, 'svm_pred': svm_pred, 'rf_pred': rf_pred
})

cnn_pred_df = pd.DataFrame({
    'participant_id': cnn_groups,
    'y_true': cnn_labels,
    'cnn_pred': cnn_preds
})

os.makedirs('outputs/predictions', exist_ok=True)
predictions_df.to_csv('outputs/predictions/baseline_predictions.csv', index=False)
cnn_pred_df.to_csv('outputs/predictions/cnn_predictions.csv', index=False)

# Identify participants all models got wrong
test_participants = sorted(set(g_test))
misclass_summary = []
for pid in test_participants:
    mask = g_test == pid
    true = y_test[mask][0]  # all same for this participant
    lr_maj = np.bincount(lr_pred[mask]).argmax()
    svm_maj = np.bincount(svm_pred[mask]).argmax()
    rf_maj = np.bincount(rf_pred[mask]).argmax()

    cnn_mask = cnn_groups == pid
    cnn_maj = np.bincount(cnn_preds[cnn_mask]).argmax() if cnn_mask.sum() > 0 else -1

    all_wrong = (lr_maj != true) and (svm_maj != true) and (rf_maj != true) and (cnn_maj != true)
    misclass_summary.append({
        'participant_id': pid,
        'true_label': le.inverse_transform([true])[0],
        'lr_pred': le.inverse_transform([lr_maj])[0],
        'svm_pred': le.inverse_transform([svm_maj])[0],
        'rf_pred': le.inverse_transform([rf_maj])[0],
        'cnn_pred': le.inverse_transform([cnn_maj])[0] if cnn_maj >= 0 else 'N/A',
        'all_wrong': all_wrong
    })

misclass_df = pd.DataFrame(misclass_summary)
print('Subject-level prediction summary:')
print(misclass_df.to_string(index=False))

hard_cases = misclass_df[misclass_df['all_wrong']]
if len(hard_cases) > 0:
    print(f'\n{len(hard_cases)} participant(s) misclassified by ALL models:')
    print(hard_cases[['participant_id', 'true_label']].to_string(index=False))
else:
    print('\nNo participants were misclassified by all models simultaneously.')

misclass_df.to_csv('outputs/predictions/subject_level_error_analysis.csv', index=False)
print('\nSaved: outputs/predictions/subject_level_error_analysis.csv')

## 22. Saving Experiment Artifacts

In [ ]:
import json

# Save experiment metadata
experiment_meta = {
    'dataset': 'OpenNeuro ds004504',
    'dataset_license': 'CC0',
    'total_participants': int(len(participants_df)),
    'loaded_participants': int(len(np.unique(groups_all))),
    'total_windows': int(X_all.shape[0]),
    'window_shape': list(X_all.shape[1:]),
    'sampling_rate': int(expected_sfreq),
    'channels': expected_channels,
    'preprocessing': preproc_config,
    'random_seed': RANDOM_SEED,
    'split': {
        'train_participants': len(train_p),
        'val_participants': len(val_p),
        'test_participants': len(test_p)
    },
    'python_version': sys.version,
    'pytorch_version': torch.__version__,
    'device': DEVICE
}

with open('outputs/experiment_metadata.json', 'w') as f:
    json.dump(experiment_meta, f, indent=2, default=str)

# Learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_title('CNN Loss Curves')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(history['val_f1'], label='Val F1', color='green')
ax2.set_title('CNN Validation F1')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Macro F1')
ax2.legend()
plt.tight_layout()
plt.savefig('outputs/figures/cnn_learning_curves.png', dpi=150)
plt.show()

# Zip all outputs for download
!zip -r /content/proteus_arc_outputs.zip outputs/
print('\n=== Experiment Complete ===')
print('Download outputs: /content/proteus_arc_outputs.zip')
print('\nSaved artifacts:')
for root, dirs, files in os.walk('outputs'):
    for f in files:
        print(f'  {os.path.join(root, f)}')